# Tagger Baseado em RNN

In [3]:
# 1. Clona o projeto para o ambiente temporário do Colab
!git clone https://github.com/lucasaamorim/NLP

Cloning into 'NLP'...
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (40/40), done.
^C
[Errno 2] No such file or directory: 'NLP'
/content/NLP
Ignoring colorama: markers 'sys_platform == "win32"' don't match your environment
  Using cached absl_py-2.4.0-py3-none-any.whl (135 kB)
  Using cached click-8.4.1-py3-none-any.whl (116 kB)
  Using cached keras-3.14.1-py3-none-any.whl (1.6 MB)
  Using cached nltk-3.9.4-py3-none-any.whl (1.6 MB)
  Using cached numpy-2.4.6.tar.gz (20.7 MB)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_inter

In [4]:
# 2. Entra na pasta do projeto
%cd NLP

[Errno 2] No such file or directory: 'NLP'
/content/NLP


In [7]:
# 3. Instala o arquivo de dependências (caso o notebook use algo além do TensorFlow padrão)
!uv pip install --no-cache -r requirements.txt

Using Python 3.12.13 environment at: /usr
Resolved 22 packages in 4.81s
Prepared 10 packages in 50m 29s
Uninstalled 10 packages in 329ms
Installed 10 packages in 94ms
 - absl-py==1.4.0
 + absl-py==2.4.0
 - click==8.4.2
 + click==8.4.1
 - keras==3.13.2
 + keras==3.14.1
 - nltk==3.9.1
 + nltk==3.9.4
 - numpy==2.0.2
 + numpy==2.4.6
 - regex==2025.11.3
 + regex==2026.5.9
 - rich==13.9.4
 + rich==15.0.0
 - scikit-learn==1.6.1
 + scikit-learn==1.9.0
 - scipy==1.16.3
 + scipy==1.17.1
 - tqdm==4.67.3
 + tqdm==4.68.2


In [10]:
import numpy as np
import keras
from keras import layers

from src.utils.data_loader import load_tagging_data
from src.utils.preprocessing import tokenize_sentences, vectorize_tags

In [13]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [16]:
%cd /content/drive/MyDrive/pos_tagging_data/data

/content/drive/MyDrive/pos_tagging_data/data


In [19]:
# ln -s /caminho/real/no/drive /content/nome_que_o_codigo_espera
!ln -s "/content/drive/MyDrive/pos_tagging_data/data" "/content/NLP/data"

###Carregando os dados de POS Tagging

In [21]:
data = load_tagging_data()
print("Dados do POS Tagging carregados")

Dados do POS Tagging carregados


###Separando os splits originais

In [22]:
train_sentences = data["train"]["sentences"]
train_tags = data["train"]["tags"]

val_sentences = data["val"]["sentences"]
val_tags = data["val"]["tags"]

print(f"Sentenças de Treino: {len(train_sentences)} | Validação: {len(val_sentences)}")

Sentenças de Treino: 38219 | Validação: 5527


###Processando o Vocabulário de Palavras

In [30]:
# vocab_size = 15000  # Limite máximo de palavras únicas
vectorizer, max_len = tokenize_sentences(train_sentences)

X_train = vectorizer([" ".join(s) for s in train_sentences]).numpy()
X_val = vectorizer([" ".join(s) for s in val_sentences]).numpy()

###Processando o Vocabulário de Tags

In [31]:
tag_lookup, Y_train = vectorize_tags(train_tags, max_len)

num_tags = tag_lookup.vocabulary_size()
Y_val_list = []
for sentence_tags in val_tags:
    tag_ids = tag_lookup(sentence_tags).numpy()
    pad_length = max_len - len(tag_ids)
    if pad_length > 0:
        padded_tags = np.pad(tag_ids, (0, pad_length), constant_values=0) # 0 é o ID do <PAD>
    else:
        padded_tags = tag_ids[:max_len]
    Y_val_list.append(padded_tags)
Y_val = np.array(Y_val_list)

print(f"Formato final de X_train: {X_train.shape}")
print(f"Formato final de Y_train: {Y_train.shape}")
print(f"Número total de tags (classes): {num_tags}")

KeyboardInterrupt: 

In [29]:
# Hiperparâmetros
embedding_dim = 128
rnn_units = 128
BATCH_SIZE = 32
EPOCHS = 5

inputs = keras.Input(shape=(max_len,), dtype="int32")

rnn = layers.Embedding(
    input_dim=vectorizer.vocabulary_size(),
    output_dim=embedding_dim,
    mask_zero=True
)(inputs)

###Construindo o modelo RNN unidirecional

In [ ]:
rnn_unidirecional = layers.SimpleRNN(rnn_units, return_sequences=True)(rnn)

outputs = layers.Dense(num_tags, activation="softmax")(rnn_unidirecional)

rnn_unidirecional_model = keras.Model(inputs, outputs)

rnn_unidirecional_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

rnn_unidirecional_model.summary()

###Construindo o modelo LSTM unidirecional


In [ ]:
lstm_unidirecional = layers.LSTM(rnn_units, return_sequences=True)(rnn)

outputs = layers.Dense(num_tags, activation="softmax")(lstm_unidirecional)

lstm_unidirecional_model = keras.Model(inputs, outputs)

lstm_unidirecional_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

lstm_unidirecional_model.summary()

###Construindo o modelo RNN bidirecional

In [ ]:
rnn_bidirecional = layers.Bidirectional(layers.SimpleRNN(rnn_units, return_sequences=True))(rnn)

outputs = layers.Dense(num_tags, activation="softmax")(rnn_bidirecional)

rnn_bidirecional_model = keras.Model(inputs, outputs)

rnn_bidirecional_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

rnn_bidirecional_model.summary()

###Construindo o modelo LSTM Bidirecional

In [25]:
# RNN Bidirecional para capturar contexto tanto da esquerda-para-direita quanto da direita-para-esquerda
lstm_bidirecional = layers.Bidirectional(layers.LSTM(rnn_units, return_sequences=True))(rnn)

# Camada densa distribuída no tempo (computa uma distribuição de probabilidade por palavra)
outputs = layers.Dense(num_tags, activation="softmax")(lstm_bidirecional)

lstm_bidirecional_model = keras.Model(inputs, outputs)

# Usamos SparseCategoricalCrossentropy porque nosso Y contém IDs inteiros (não one-hot vectors)
lstm_bidirecional_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

lstm_bidirecional_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 249)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 249, 128)  │  1,920,000 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 249)       │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 249, 256)  │    263,168 │ embedding[0][0],  │
│ (Bidirectional)     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 249, 71)   │     18,247 │ bidirectional[0]… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,201,415 (8.40 MB)

 Trainable params: 2,201,415 (8.40 MB)

 Non-trainable params: 0 (0.00 B)

### Treinamento da RNN Unidirecional

In [ ]:
history = rnn_unidirecional_model.fit(
    X_train,
    Y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val, Y_val)
)

### Treinamento da LSTM Unidirecional

In [ ]:
history = lstm_unidirecional_model.fit(
    X_train,
    Y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val, Y_val)
)

### Treinamento da RNN Bidirecional

In [ ]:
history = rnn_bidirecional_model.fit(
    X_train,
    Y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val, Y_val)
)

### Treinamento da LSTM Bidirecional

In [27]:
history = lstm_bidirecional_model.fit(
    X_train,
    Y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_val, Y_val)
)

Epoch 1/5


KeyboardInterrupt: 